# 02 – Detect Eye Blinks

Eye blinks create large-amplitude artifacts (100–300 µV) on **frontal channels** (ch1–ch4).

This notebook:
1. Loads a session (CSV or live WebSocket)
2. Bandpass-filters the frontal channels (1–10 Hz to isolate blink band)
3. Detects blinks with a simple amplitude threshold
4. Plots detected blinks on the raw trace

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal

SAMPLE_RATE = 250
FRONTAL_CHANNELS = ["ch1", "ch2"]  # adjust for your montage

## Load data

In [ ]:
import glob

csv_files = sorted(glob.glob("../recordings/pieeg_*.csv"))
df = pd.read_csv(csv_files[-1])
df["time"] = df["timestamp"] - df["timestamp"].iloc[0]
print(f"Loaded {len(df)} samples  ({df['time'].iloc[-1]:.1f} s)")

## (Alternative) Stream live from PiEEG server

Uncomment the cell below if you want to detect blinks in real time instead.

In [ ]:
# import asyncio, json, websockets
#
# async def collect(seconds=10):
#     rows = []
#     async with websockets.connect("ws://localhost:1616") as ws:
#         welcome = json.loads(await ws.recv())
#         n_ch = welcome["channels"]
#         target = seconds * SAMPLE_RATE
#         while len(rows) < target:
#             msg = json.loads(await ws.recv())
#             if "channels" in msg:
#                 rows.append([msg["t"]] + msg["channels"])
#     cols = ["timestamp"] + [f"ch{i+1}" for i in range(n_ch)]
#     return pd.DataFrame(rows, columns=cols)
#
# df = await collect(10)
# df["time"] = df["timestamp"] - df["timestamp"].iloc[0]

## Bandpass filter (1–10 Hz) to isolate blink band

In [ ]:
sos = signal.butter(4, [1.0, 10.0], btype="band", fs=SAMPLE_RATE, output="sos")

filtered = {}
for ch in FRONTAL_CHANNELS:
    filtered[ch] = signal.sosfiltfilt(sos, df[ch].values)

## Detect blinks by threshold

A blink is a peak in the filtered signal that exceeds **3× the standard deviation**.
We also enforce a minimum 300 ms gap between consecutive blinks.

In [ ]:
def detect_blinks(data, fs, threshold_std=3.0, min_gap_s=0.3):
    """Return indices of detected blinks."""
    threshold = np.std(data) * threshold_std
    peaks, props = signal.find_peaks(np.abs(data), height=threshold,
                                      distance=int(min_gap_s * fs))
    return peaks

# Detect on the first frontal channel
blink_ch = FRONTAL_CHANNELS[0]
blink_idx = detect_blinks(filtered[blink_ch], SAMPLE_RATE)
blink_times = df["time"].values[blink_idx]

print(f"Detected {len(blink_idx)} blink(s) on {blink_ch}")
if len(blink_idx) > 0:
    rate = len(blink_idx) / df["time"].iloc[-1] * 60
    print(f"Blink rate: {rate:.1f} blinks/min")

## Plot blinks on raw trace

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 5), sharex=True)

# Raw
ax1.plot(df["time"], df[blink_ch], linewidth=0.4, label="Raw")
ax1.scatter(blink_times, df[blink_ch].values[blink_idx],
            color="red", s=40, zorder=5, label="Blink")
ax1.set_ylabel(f"{blink_ch} (µV)")
ax1.legend(loc="upper right")
ax1.set_title("Raw signal with detected blinks")

# Filtered
ax2.plot(df["time"], filtered[blink_ch], linewidth=0.5, color="darkorange")
ax2.axhline(np.std(filtered[blink_ch]) * 3, color="red", ls="--", lw=0.8, label="Threshold")
ax2.axhline(-np.std(filtered[blink_ch]) * 3, color="red", ls="--", lw=0.8)
ax2.scatter(blink_times, filtered[blink_ch][blink_idx],
            color="red", s=40, zorder=5)
ax2.set_ylabel("Filtered (µV)")
ax2.set_xlabel("Time (s)")
ax2.legend(loc="upper right")
ax2.set_title("Bandpass 1–10 Hz")

plt.tight_layout()
plt.show()

## Overlay blink epochs (ERP-style)

Stack 500 ms windows centered on each blink to see the average blink waveform.

In [ ]:
HALF_WIN = int(0.25 * SAMPLE_RATE)  # 250 ms each side
raw = df[blink_ch].values

epochs = []
for idx in blink_idx:
    if idx - HALF_WIN >= 0 and idx + HALF_WIN < len(raw):
        epochs.append(raw[idx - HALF_WIN : idx + HALF_WIN])

if epochs:
    epochs = np.array(epochs)
    t_epoch = np.linspace(-250, 250, epochs.shape[1])  # ms

    fig, ax = plt.subplots(figsize=(8, 4))
    for ep in epochs:
        ax.plot(t_epoch, ep, color="steelblue", alpha=0.3, lw=0.6)
    ax.plot(t_epoch, epochs.mean(axis=0), color="black", lw=2, label="Mean")
    ax.axvline(0, color="red", ls="--", lw=0.8)
    ax.set_xlabel("Time from blink peak (ms)")
    ax.set_ylabel("Amplitude (µV)")
    ax.set_title(f"Blink Epochs – {blink_ch} (n={len(epochs)})")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No valid blink epochs found (signal too short or no blinks detected).")